In [23]:
import os
import csv
import json
import oracledb
from langchain_core.documents import Document
from langchain_oracledb.vectorstores import oraclevs
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_oracledb.vectorstores.oraclevs import OracleVS
from langchain_core.vectorstores.base import VectorStoreRetriever
from langchain_community.vectorstores.utils import DistanceStrategy

In [24]:
username = "system"
password = "oracle"
dsn = "192.168.1.248:1521/FREEPDB1"

In [25]:
try:
    connection = oracledb.connect(user=username, password=password, dsn=dsn)
    print("Connection successful!")

except oracledb.Error as e:
    error_obj, = e.args
    print(f"Oracle Error: {error_obj.message}")

except Exception as e:
    print(f"Undefined Error: {e}")

Connection successful!


In [26]:
corpus_path = os.path.join(os.path.dirname(os.getcwd()), "scifact", "corpus.jsonl")
corpus_path

'd:\\IE103_Final_Project\\scifact\\corpus.jsonl'

In [27]:
documents_langchain = []

with open(corpus_path, "r", encoding="utf-8") as document_jsonl_list:
    for line in document_jsonl_list:
        doc = json.loads(line)
        metadata = {"id": doc["_id"], "title": doc["title"]}
        doc_langchain = Document(page_content=doc["text"], metadata=metadata)
        documents_langchain.append(doc_langchain)

In [28]:
model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3898.09it/s]


In [29]:
vector_store = OracleVS(
    client=connection,
    embedding_function=model,
    table_name="SciFact",
    distance_strategy=DistanceStrategy.COSINE,
)

In [30]:
oraclevs.create_index(
    client=connection,
    vector_store=vector_store,
    params={"idx_name": "hnsw_scifact", "idx_type": "HNSW"},
)

In [31]:
def RetrieveTopK(vector_store: OracleVS, k: int) -> VectorStoreRetriever:
    retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k}
    )
    return retriever

In [32]:
retriever = RetrieveTopK(vector_store, 10)

In [33]:
retrieval_results = retriever.invoke("0-dimensional biomaterials show inductive properties.")
retrieval_results

[Document(id='6521069a-83f0-409e-8e6c-b32c45bd9824', metadata={'id': '6521069a-83f0-409e-8e6c-b32c45bd9824', 'title': 'Nonlinear Elasticity in Biological Gels', 'corpus_id': '4346436'}, page_content='Unlike most synthetic materials, biological materials often stiffen as they are deformed. This nonlinear elastic response, critical for the physiological function of some tissues, has been documented since at least the 19th century, but the molecular structure and the design principles responsible for it are unknown. Current models for this response require geometrically complex ordered structures unique to each material. In this Article we show that a much simpler molecular theory accounts for'),
 Document(id='b3c1a2a0-5dfa-40a5-855b-0eade8f9a378', metadata={'id': 'b3c1a2a0-5dfa-40a5-855b-0eade8f9a378', 'title': 'Complex Tissue and Disease Modeling using hiPSCs.', 'corpus_id': '29638116'}, page_content='Defined genetic models based on human pluripotent stem cells have opened new avenues f

In [34]:
def ChunksToDocuments(retrieval_results: list[Document], documents_langchain: list[Document]) -> list[Document]:
    corpus_results = set()
    for result in retrieval_results:
        corpus_id = result.metadata["corpus_id"]
        corpus_results.add(corpus_id)

    document_results = []
    for document in documents_langchain:
        corpus_id = document.metadata["id"]
        if corpus_id in corpus_results:
            document_results.append(document)

    return document_results

In [35]:
document_results = ChunksToDocuments(retrieval_results, documents_langchain)
document_results

[Document(metadata={'id': '3874000', 'title': 'Tissue Mechanics Orchestrate Wnt-Dependent Human Embryonic Stem Cell Differentiation.'}, page_content='Regenerative medicine is predicated on understanding the mechanisms regulating development and applying these conditions to direct stem cell fate. Embryogenesis is guided by cell-cell and cell-matrix interactions, but it is unclear how these physical cues influence stem cells in culture. We used human embryonic stem cells (hESCs) to examine whether mechanical features of the extracellular microenvironment could differentially modulate mesoderm specification. We found that, on a hydrogel-based compliant matrix, hESCs accumulate β-catenin at cell-cell adhesions and show enhanced Wnt-dependent mesoderm differentiation. Mechanistically, Src-driven ubiquitination of E-cadherin by Cbl-like ubiquitin ligase releases P120-catenin to facilitate transcriptional activity of β-catenin, which initiates and reinforces mesoderm differentiation. By contr

In [36]:
def DocumentsToCorpusID(document_results: list[Document]) -> list[str]:
    corpus_ids = []
    for document in document_results:
        corpus_id = document.metadata["id"]
        corpus_ids.append(corpus_id)
    
    return corpus_ids

In [37]:
corpus_ids = DocumentsToCorpusID(document_results)
corpus_ids

['3874000',
 '4346436',
 '4427392',
 '6280907',
 '6863070',
 '7583104',
 '8891333',
 '25404036',
 '29638116',
 '31715818']

In [38]:
test_path = os.path.join(os.path.dirname(os.getcwd()), "scifact", "qrels", "test.tsv")
test_path

'd:\\IE103_Final_Project\\scifact\\qrels\\test.tsv'

In [39]:
test = {}

with open(test_path, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f, delimiter='\t')

    for row in reader:
        query_id = row["query-id"]
        corpus_id = row["corpus-id"]
        test[query_id] = corpus_id

In [40]:
for query_id, corpus_id in test.items():
    print(f"Query ID: {query_id}. Corpus ID: {corpus_id}.")
    break

Query ID: 1. Corpus ID: 31715818.


In [41]:
queries_path = os.path.join(os.path.dirname(os.getcwd()), "scifact", "queries.jsonl")
queries_path

'd:\\IE103_Final_Project\\scifact\\queries.jsonl'

In [42]:
queries = {}

with open(queries_path, "r", encoding="utf-8") as document_jsonl_list:
    for line in document_jsonl_list:
        doc = json.loads(line)
        query_id = doc["_id"]
        text = doc["text"]
        queries[query_id] = text

In [43]:
for query_id, query_text in queries.items():
    print(f"Query ID: {query_id}. Query text: {query_text}")
    break

Query ID: 0. Query text: 0-dimensional biomaterials lack inductive properties.


In [44]:
for k in range(5, 51, 5):
    correct = 0
    retriever = RetrieveTopK(vector_store, k)

    for i, (query_id, corpus_id) in enumerate(test.items()):
        retrieval_results = retriever.invoke(queries[query_id])
        document_results = ChunksToDocuments(retrieval_results, documents_langchain)
        corpus_ids = DocumentsToCorpusID(document_results)

        if corpus_id in corpus_ids:
            correct += 1

    print(f"Hit@{k}: {correct / 300 * 100:.2f}%")

Hit@5: 73.33%
Hit@10: 78.67%
Hit@15: 81.33%
Hit@20: 83.33%
Hit@25: 84.67%
Hit@30: 85.33%
Hit@35: 86.33%
Hit@40: 86.67%
Hit@45: 88.00%
Hit@50: 88.67%
